# PII Shield — Interactive Demonstration

**PII Shield** is a 100% local, privacy-preserving pipeline that extracts text from enterprise business documents (scanned PDFs, rich DOCX, slide PPTX), detects and classifies PII across multiple extraction and detection layers, redacts it deterministically, and produces an audit report before text ever reaches a Large Language Model (LLM).

### Core Principles
1. **Zero External Calls**: 100% local inference (Presidio, spaCy, Tesseract, Regex). No telemetry, no cloud APIs.
2. **Fail-Closed Safety**: Low OCR confidence or corrupt files emit BLOCKED chunks, preventing unsafe text downstream.
3. **Zero Raw PII Exposure**: Audit reports only contain masked values (e.g. `J*****h`), never raw sensitive data.
4. **Reversible & Consistent**: Identical entities map to identical placeholder tokens (`[PERSON_1]`, `[EMAIL_1]`) via an isolated, encrypted mapping store.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd

with open("config.yaml", "r", encoding="utf-8") as fh:
    config = yaml.safe_load(fh)

print(f"Loaded PII Shield config with profile: {config.get('active_profile')}")
print(f"Enabled entity types: {len(config.get('enabled_entities', []))}")

## 1. Document Extraction & Routing
The router checks magic bytes and extensions to dispatch to the appropriate parser (PyMuPDF with deskew/adaptive OCR, python-docx with table and image extraction, python-pptx with notes).

In [ ]:
from extract.router import route

sample_file = str(Path("synthetic/training_guide.docx").resolve())
chunks = route(sample_file, config)

print(f"Extracted {len(chunks)} chunks from {Path(sample_file).name}")
blocked_count = sum(1 for c in chunks if c.is_blocked())
print(f"Blocked chunks: {blocked_count}")

# Show first few chunks
for c in chunks[:5]:
    print(f"[{c.location}] ({c.method}): {c.text[:80]}...")

## 2. Multi-Layer PII Detection
Detection combines:
- **Regex Rules + Mathematical Validators** (Luhn for Credit Cards, Verhoeff for Aadhaar, mod-97 for IBAN, SSN area sanity)
- **Local Named Entity Recognition** (Presidio + spaCy `en_core_web_lg`)
- **Context Engine** (Column-header boost, labelled prefixes like 'Approved by:')
- **Merge & Conflict Resolver** (Greedy overlap resolution & span union)

In [ ]:
from detect.merge import detect_chunk, Entity

all_entities = []
for c in chunks:
    if not c.is_blocked():
        ents = detect_chunk(c, config)
        all_entities.extend(ents)

print(f"Total PII entities detected: {len(all_entities)}")

# Display sample detected entities in a safe DataFrame (Masked only)
df_entities = pd.DataFrame([
    {
        "Type": e.entity_type,
        "Masked Value": e.masked_value,
        "Sensitivity": e.sensitivity,
        "Confidence": f"{e.confidence:.2f}",
        "Methods": ", ".join(e.methods),
        "Location": e.location
    }
    for e in all_entities[:10]
])
df_entities

## 3. Redaction & Mapping Store
Entities are replaced using consistent placeholder tokens (`[PERSON_1]`, `[EMAIL_1]`).
Raw-to-placeholder mappings are stored in a separate file excluded from the audit log.

In [ ]:
from redact.mapping import PlaceholderMapping
from redact.redactor import redact_chunks

mapping = PlaceholderMapping(source_file=sample_file, output_dir="output")
redacted_chunks = redact_chunks(chunks, all_entities, mapping, config)

# Inspect sample redacted text
for c in redacted_chunks[:5]:
    if not c.is_blocked() and any(k in c.text for k in ["[", "]"]):
        print(f"--- {c.location} ---")
        print(c.text)
        print()

## 4. Verification & Leakage Gate
The detector is re-run on redacted content. If any unredacted PII is discovered, `downstream_safe` is set to `False`.

In [ ]:
from verify.leakage import check_leakage

leak_res = check_leakage(redacted_chunks, config)
print(leak_res.summary())
print(f"Downstream LLM Safe: {leak_res.downstream_safe}")

## 5. Audit Report Generation
Generates structured audit logs (CSV, JSON, and summary table) strictly without exposing raw values.

In [ ]:
from report.report import build_file_summary, generate_report

summary = build_file_summary(sample_file, chunks, all_entities, leak_result=leak_res, structure_score=None)
print(f"File: {summary['file']}")
print(f"Risk Level: {summary['risk_level']}")
print(f"Total PII: {summary['pii_count']}")
print(f"Breakdown: {summary['pii_by_type']}")